In [11]:
!pip install -q chromadb
!pip install -U -q "google-genai"

## RAG con LangChain
Ejecutamos el cuaderno con las funciones y variables necesarias para añadir datos y consultar la base de datos de Chroma utilizando la interfaz vector store de LangChain. 

In [2]:
%run ./RAG_LangChain.ipynb

---
<h2>Creacion de Agentes con Google Agent Development Kit</h2>


In [3]:
# Instalar librerías necesarias
!pip install google-adk litellm -q

import os
import requests
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# API Key y configuración
#os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"
MODEL_GEMINI_2_0_FLASH = "gemini-2.0-flash"

print("Configuración ADK completada.")



"pip" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


Configuración ADK completada.


In [4]:
def obtener_peliculas_por_titulo(titulo: str):
    return query_col(query=titulo, collection='movies')[0]['metadata']

def obtener_detalles_pelicula(titulo: str):
    info = obtener_peliculas_por_titulo(titulo)
    if isinstance(info, dict) and 'movie_title' in info:
        return {
            "titulo": info.get("movie_title"),
            "director": info.get("director"),
            "cast": info.get("cast"),
            "fecha_estreno": info.get("release_date")
        }
    return info  # Devuelve Wikipedia o mensaje de error

def obtener_sinopsis_pelicula(titulo: str):
    info = obtener_peliculas_por_titulo(titulo)
    if isinstance(info, dict) and 'overview' in info:
        return info['overview']
    # Si no hay sinopsis, mirar Wikipedia
    wiki = search_in_wikipedia(titulo, True)
    if isinstance(wiki, dict):
        return wiki.get("summary", "Sinopsis no disponible.")
    return f"No se encontró información sobre '{titulo}'."

def obtener_detalles_actor(nombre: str):
    """
    Obtiene información detallada de un actor por su nombre.
    Primero intenta en TMDB y, si no encuentra resultados, busca en Wikipedia.
    """
    # 1 Buscar actor en TMDB
    url_search = f"https://api.themoviedb.org/3/search/person?query={nombre}&include_adult=false&language=en-US&page=1"
    response_search = requests.get(url_search, headers=TMDB_HEADERS)
    data_search = response_search.json()

    if data_search.get('results'):
        # Tomamos el primer resultado
        actor = data_search['results'][0]
        person_id = actor['id']

        # 2 Obtener detalles del actor usando la función existente
        detalles = get_person_details(person_id)
        gender = "Not set"
        if detalles['gender'] == 1:
            gender = "female"
        elif detalles['gender'] == 2:
            gender = "male"
        elif detalles['gender'] == 3:
            gender = "non binary"
        return {
            "nombre": detalles.get('name'),
            "biografia": detalles.get('biography') or "Sin biografía disponible",
            "fecha_nacimiento": detalles.get('birthday'),
            "fecha_fallecimiento": detalles.get('deathday'),
            "lugar_nacimiento": detalles.get('place_of_birth'),
            "departamento": detalles.get('known_for_department'),
            "genero": gender
        }

    else:
        #  3 Fallback a Wikipedia si no se encuentra en TMDB
        wiki = search_in_wikipedia(nombre)
        if isinstance(wiki, dict):
            return {
                "nombre": nombre,
                "biografia": wiki.get("summary", "Sin biografía disponible"),
                "url_wikipedia": wiki.get("url")
            }
        else:
            return f"No se encontró información sobre '{nombre}'."

    


<h2>Agente Cine</h2>

In [5]:
# Crear agente con ADK
researcher_agent = Agent(
    name="researcher_agent",
    model=MODEL_GEMINI_2_0_FLASH,
    description="Agente experto en actualizar la base de datos con información que necesite el usuario.",
    instruction=(
    """
    Eres un asistente experto en buscar información sobre películas.
    Dispones de varias herramientas para añadir información a tu base de datos:
    - add_movies_to_collection: Busca infomación de una película a partir de su título (resumen, fecha de salida, cast...).
    - add_reviews_to_collection: Busca información sobre la opinión del público, reseñas, a partir del título de una película.
    - add_person_to_collection: Busca información detallada sobre actores, directores y otros miembros del cast a partir de su nombre.
        
    Deberás sacar la información que el usuario necesite a partir de su pregunta, por ejemplo:
        - Si pregunta sobre los actores en una película, añades información sobre el cast con add_movies_to_collection.
        - Si pregunta sobre algún actor en concreto, añades información con add_person_to_collection.
    NO respondes la pregunta del usuario, simplemente añades información y le dices lo que acabas de hacer.
    """
    ),
    tools=[add_movies_to_collection, add_reviews_to_collection, add_person_to_collection]
)

print(f"Agente '{researcher_agent.name}' creado con modelo '{MODEL_GEMINI_2_0_FLASH}'.")


Agente 'researcher_agent' creado con modelo 'gemini-2.0-flash'.


<h2>Agente Actores</h2>

In [6]:
# Crear agente experto en actores
query_agent = Agent(
    name="query_agent",
    model=MODEL_GEMINI_2_0_FLASH,
    description="Agente experto en preguntar información a la base de datos dependiendo en la pregunta del usuario.",
    instruction=(
    """
    Eres un agente experto en cine. Tu función es, utilizando la pregunta del usuario, preguntar a la base de datos
    por toda la información relevante para el usuario.
    Dispones de una herramienta para consultar tu base de datos:
    - query_col: Consultas una de tus colecciones ('movies', 'reviews' o 'people') dependiendo en la información
    que necesites para satisfacer la pregunta del usuario:
        - 'movies' tiene información general sobre películas, fecha de salida, el cast de la película, descripción.
        - 'reviews' tiene la opiniones, reviews, críticas sobre las películas.
        - 'people' tiene información sobre el cast, actores, gente del mundo del cine.
    query_col permite buscar una película en concreto según su año de salida introduciendo otro parámetro más. Un ejemplo
    de llamada a la tool sabiendo el año de salida sería este:
        query_col('dead people', 'movies', '2001')

    Si no encuentras información sobre el tema preguntado por el usuario o no es relevante para tu función, dilo.
    """
    ),
    tools=[query_col]
)

print(f"Agente '{query_agent.name}' creado con modelo '{MODEL_GEMINI_2_0_FLASH}'.")


Agente 'query_agent' creado con modelo 'gemini-2.0-flash'.


In [7]:
# Crear Runner y sesión
session_service = InMemorySessionService()
APP_NAME = "cine_app"
USER_ID = "Usuario"
SESSION_ID = "user"

import asyncio

# Crear sesión async
session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)
print(f"Session creada: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

#agente de cine
researcher_runner = Runner(
    agent=researcher_agent,
    app_name=APP_NAME,
    session_service=session_service
)
print(f"Runner creado para el agente '{researcher_runner.agent.name}'.")

#agente de actores
query_runner = Runner(
    agent=query_agent,
    app_name=APP_NAME,
    session_service=session_service
)

print(f"Runner creado para el agente '{query_runner.agent.name}'.")



Session creada: App='cine_app', User='Usuario', Session='user'
Runner creado para el agente 'researcher_agent'.
Runner creado para el agente 'query_agent'.


<h3>Funcion para llamar a cualquier agente</h3>

In [8]:
# Función genérica para llamar a cualquier agente
async def call_any_agent_async(query: str, runner: Runner):
    print(f"\n>>> User Query: {query}")
    content = types.Content(role='user', parts=[types.Part(text=query)])
    final_response_text = "El agente no produjo respuesta final."

    async for event in runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=content):
        if event.is_final_response():
            if event.content and event.content.parts:
                final_response_text = event.content.parts[0].text
            break

    print(f"<<< Agent Response: {final_response_text}")


In [12]:
#hablar con el agente
# await call_any_agent_async("de qué año es 'los otros'?",runner=researcher_runner)
await call_any_agent_async("qué me puedes decir sobre el mago de oz de 1930?",runner=query_runner)



>>> User Query: qué me puedes decir sobre el mago de oz de 1930?
<<< Agent Response: No tengo información sobre ninguna película del Mago de Oz de 1930. Puedo buscar información sobre otras películas del Mago de Oz si quieres.



---
<h1>Creacion Multi-Agentes</h1>
<br><h3>1º Instalar ADK</h3>

In [5]:
# Instalar ADK y LiteLLM para soporte multi-model 

!pip install google-adk -q
!pip install litellm -q

print("Instalacion completada.")

Instalacion completada.


<h3>2º Importar librerias</h3>

In [6]:

import os
import asyncio
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm #Para el soporte multi-model
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types # Para crear mensage 

import warnings
# Ignorar warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.ERROR)

print("Librerias importadas.")

Librerias importadas.


<h3>3º Importar API Keys</h3>

In [8]:



from dotenv import load_dotenv

load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv('GEMINI_API_KEY')

# --- Comprobar API keys (Opcional) ---
print("API Keys:")
print(f"Google API Key : {'Si' if os.environ.get('GOOGLE_API_KEY') and os.environ['GOOGLE_API_KEY'] != 'YOUR_GOOGLE_API_KEY' else 'No (REPLACE PLACEHOLDER!)'}")
print(f"OpenAI API Key : {'Si' if os.environ.get('OPENAI_API_KEY') and os.environ['OPENAI_API_KEY'] != 'YOUR_OPENAI_API_KEY' else 'No '}")
print(f"Anthropic API Key : {'Si' if os.environ.get('ANTHROPIC_API_KEY') and os.environ['ANTHROPIC_API_KEY'] != 'YOUR_ANTHROPIC_API_KEY' else 'No '}")

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"


API Keys:
Google API Key : Si
OpenAI API Key : No 
Anthropic API Key : No 


<h3>4º Definir el modelo de Gemini</h3>

In [9]:
MODEL_GEMINI_2_0_FLASH = "gemini-2.0-flash"
print("\nEntorno configurado correctamente.")


Entorno configurado correctamente.


<h2>Definir el agente principal</h2>
<br><h3>1º Definir las Tools</h3>